In [ ]:
import math
import time
import numpy as np
from numpy.linalg import norm
from numpy.linalg import inv
import matplotlib.pyplot as plt
import sys

# Aufgabe 6: CG-Verfahren und PCG-Verfahren

Wir betrachten das lineare Gleichungssystem $Ax = b$, wobei
$$
 A =  \begin{bmatrix}
        \phantom{-}2 & -1 &  &  &  \\
        -1& \phantom{-}2 & -1&  & \\
        & \ddots & \ddots & \ddots & \\
        & & -1 & \phantom{-}2 & -1 \\
        & & & -1 & \phantom{-}2 
    \end{bmatrix} \in \mathbb{R}^{n\times n}.
$$
Dieses System wollen wir mithilfe des **CG-Verfahrens** (konjugierte Gradienten) und des **PCG-Verfahrens** (vorkonditionierte Gradienten) lösen.

## Teil 1: Implementierung der Hilfsfunktionen

Für großes $n$ ist es sehr ineffizient, die Matrix $A$ und den Präkonditionierer $S$ zu speichern. Daher wollen wir die Anwendung dieser Matrizen direkt als Funktionen implementieren. Dieser Ansatz wird auch als "Matrix-Free" bezeichnet.

* `Av = mul_A(v)`: Diese Funktion soll die Multiplikation der Matrix $A$ mit einem beliebigen Vektor $v$ effizient realisieren, **ohne** dass $A$ als $n \times n$-Matrix gespeichert werden muss.
* `w = mul_pre(r)`: Analog soll diese Funktion die Multiplikation eines Vektors $r$ mit dem Vorkonditionierer $M^{-1} = S^{-1} S^{-T}$ realisieren.

Für die Vorkonditionierung soll, wie in der Vorlesung besprochen, $S = \frac{1}{2} D + L^T$ verwendet werden, wobei die Zerlegung $A = L + D + L^T$ zugrunde liegt ($L$ ist der strikte untere Dreiecksteil und $D$ die Diagonale von $A$).

$S^{-T}S^{-1}$ kann dann leicht mittel Vorwärts- und Rückwärtssubstitution angewendet werden.

In [ ]:
def mul_A(v):
    """
    A*v = Av_mul(v)
    Legen Sie hier nicht die Matrix A an!
    """
    Av = 2.0*v
    ???
    return Av

In [ ]:
def mul_pre(r):
    """
    Anwendung des Vorkonditionierers S^(-T)S^(-1)
    (Auch hier dürfen weder A noch S^(-1) als Matrix angelegt werden!)
    """

    # Vorwärtssubstitution
    ???
    # Rückwärtssubstitution
    ???

Sie können die folgenden beideln Zellen zum Testen Ihrer Funktionen verwenden:

In [ ]:
m = 4
n = 16
A = 2.0 * np.eye(n) - (np.diag(np.ones(n-1), k=-1) + np.diag(np.ones(n-1), k=1))    # zum Testen können wir die volle Matrix erzeugen, aber nur hier!
x = np.ones(n)
# zusätzliche Werte, um den Test aussagekräftiger zu machen
x[1]  = 2
x[2]  = 5
x[-1] = 10
print(A @ x)
print(mul_A(x))
assert (A @ x == mul_A(x)).all()

In [ ]:
m = 4
n = 16
S =  np.eye(n) - np.diag(np.ones(n-1), k=1)     # zum Testen können wir die volle Matrix erzeugen, aber nur hier!
b = np.ones(n)
print(inv(S) @ inv(S.T) @ b)
print(mul_pre(b))
assert (inv(S) @ inv(S.T) @ b == mul_pre(b)).all()

### Teil 2: Implementierung der CG-Verfahren

Als Nächstes werden die Hauptfunktionen für die iterativen Lösungsverfahren benötigt:

* `solution, residuals = cg(b, mul_A, tol, max_iter)`
* `solution, residuals = pcg(b, mul_A, mul_pre, tol, max_iter)`

Diesen Funktionen werden `mul_A` und `mul_pre` als Parameter übergeben, um die Matrixoperationen durchzuführen.

Parameter:
* `tol` ist die Fehlertoleranz. Die Verfahren sollen abgebrochen werden, wenn **sowohl** die Norm des Residuums **als auch** die Norm der Differenz aufeinanderfolgender Iterierter $\leq$ `tol` ist.
* `max_iter` ist die maximale Anzahl an Iterationen. Die Verfahren sollen spätestens terminieren, wenn diese Grenze erreicht ist.
* Als Startvektor wählen Sie die rechte Seite $b$.


Die Rückgabewerte der Funktionen `cg` und `pcg` sollen sein:
* `solution`: Die vom jeweiligen Verfahren berechnete Lösung $x^{(n)}$, wobei $n$ die Anzahl benötigter Iterationen ist.
* `residuals` (Vektor): Der Vektor mit den Normen des Residuums $\| b - A x^{(i)} \|_{2}$ für jeden Iterationsschritt $i = 1, \dots, n$.


In [ ]:
def cg(b, mul_A, tol, max_iter):
    """
    Konjugierte-Gradienten-Verfahren (CG)

    Parameter:
      b         : Rechte Seite und Initialwert (Startvektor x0);
      mul_A     : Funktion, die A*v berechnet;
      tol       : Geforderte absolute Genauigkeit;
      max_iter  : Maximale Zahl von Iterationen.

    Resultate:
      solution  : Näherungslösung x^(k) von A*x=b;
      residuals : Folge der Residuennormen ||b - A*x^(i)||_2.

    Das Verfahren bricht ab, wenn entweder die maximale Zahl von
    Iterationsschritten 'max_iter' erreicht ist oder die 2-Normen von
    Residuum (A*x - b) und Schrittweite (x^(i+1) - x^(i)) beide kleiner-gleich
    der Toleranz 'tol' sind.
    """
    n = b.shape[0]
    residuals = np.zeros(max_iter + 1)
    
    iter_count = 0
    solution = np.copy(b)
    residual = b - mul_A(solution)
    search_direction = np.copy(residual)
    gamma_old = np.inner(residual, residual)
    residuals[iter_count] = math.sqrt(gamma_old) 

    while ???
        ???

    # Die resultierenden Vektoren sollen genau die richtige Länge haben;
    # alle überschüssigen Elemente werden abgeschnitten:
    residuals = residuals[:iter_count + 1]

    return solution, residuals

In [ ]:
def pcg(b, mul_A, mul_pre, tol, max_iter):
    """
    Vorkonditioniertes Konjugierte-Gradienten-Verfahren (PCG)

    Parameter:
      b         : Rechte Seite und Initialwert (Startvektor x0);
      mul_A     : Funktion, die A*v berechnet;
      mul_pre   : Funktion, die M^(-1)*v berechnet (M^(-1) = S^(-T)*S^(-1));
      tol       : Geforderte absolute Genauigkeit;
      max_iter  : Maximale Zahl von Iterationen.

    Resultate:
      solution  : Näherungslösung x^(k) von A*x=b;
      residuals : Folge der Residuennormen ||b - A*x^(i)||_2.

    Das Verfahren bricht ab, wenn entweder die maximale Zahl von
    Iterationsschritten 'max_iter' erreicht ist oder die 2-Normen von
    Residuum (A*x - b) und Schrittweite (x^(i+1) - x^(i)) beide kleiner-gleich
    der Toleranz 'tol' sind.
    """
    n = b.shape[0]
    residuals = np.zeros(max_iter + 1)

    # Initialisierung
    iter_count = 0
    solution = np.copy(b)
    residual = b - mul_A(solution)
    preconditioned_residual = mul_pre(residual)
    search_direction = np.copy(preconditioned_residual) 
    gamma_old = np.dot(preconditioned_residual, residual) 
    residuals[iter_count] = norm(residual) 

    while ???
        ???

    # Die resultierenden Vektoren sollen genau die richtige Länge haben;
    # alle überschüssigen Elemente werden daher abgeschnitten:
    residuals = residuals[:iter_count + 1]

    return solution, residuals

### Teil 3: Vergleich der Verfahren

Nun wollen wir die implementierten Verfahren vergleichen. Dafür betrachten wir verschieden große Matrizen $A \in \mathbb{R}^{n \times n}$, wobei wir $n = 2^m + 1$ setzen. Für $m = 3$ ist $A$ also eine $9 \times 9$-Matrix, für $n=10$ hingegen schon $1025 \times 1025$. 

Testen Sie ihre Verfahren, in dem Sie die folgende Zelle ausführen:

In [ ]:
from util.plotting_06 import compute_and_plot_06

tol = 1.0e-9
m_smallest = 3
m_largest  = 12

n_iterations = np.zeros((m_largest - m_smallest + 1, 2), dtype=int)
residuals = np.zeros((m_largest - m_smallest + 1, 2))
norms = np.zeros((m_largest - m_smallest + 1, 2))
times = np.zeros((m_largest - m_smallest + 1, 3))

for i, m in enumerate(range(m_smallest, m_largest + 1)):
    (
        cg_solution,
        cg_residuals,
        norms[i, 0],
        times[i, 0],
        pcg_solution,
        pcg_residuals,
        norms[i, 1],
        times[i, 1],
    ) = compute_and_plot_06(mul_A, mul_pre, cg, pcg, m, tol)
    n_iterations[i, 0] = cg_residuals.shape[0]
    n_iterations[i, 1] = pcg_residuals.shape[0]
    residuals[i, 1] = pcg_residuals[-1]
    residuals[i, 0] = cg_residuals[-1]

# Ausgabe der Tabellen
names = ["CG", "PCG"]
for i in range(2):
    print(f"{names[i]}-Verfahren")
    print('m\titer\t||x-u||\t\t||b-Ax||\ttime[s]')
    for j, m in enumerate(range(m_smallest, m_largest + 1)):
        print(f"{m}\t{n_iterations[j,i]}\t{format(norms[j, i], '.2e')}\t{format(residuals[j, i], '.2e')}\t{format(times[j, i], '.1e')}\t")
    print("")